# hand_matchup 추가 — 유일하게 채택된 matchup 피처

로직은 [catboost_handmatchup.py](catboost_handmatchup.py)에 있고, 이 노트북은 그걸 불러와서 실행 + 결과 해석을 남긴다.

`team_matchup`(투수팀 x 타자팀), `hand_matchup`(투수손 x 타자손), `count_state`(볼카운트) 3종을 각각 baseline 위에 단독으로 얹어봤는데, `hand_matchup`만 rolling OOT(fixed-iteration)에서 primary(2024)와 2022 양쪽에서 일관되게 baseline을 앞섰다. `team_matchup`/`count_state`는 확실히 폐기(HANDOFF.md 실험 로그 참고).

처음 single-split 결과(+11.23)만 보고는 노이즈 범위 안이라 보류했었는데, 재확인 과정에서 `hand_matchup_oot.py`(폴드별 독립 조기종료)가 방법론 버그였다는 걸 발견했다 — `baseline_catboost.rolling_oot_evaluate()`의 docstring에 이미 "폴드별 독립 조기종료는 2023 폴드를 무너뜨린다"고 경고돼 있던 걸 놓쳤다. `train_catboost_fixed` + `rolling_oot_evaluate_fixed`로 다시 돌리니 결과가 뒤집혔다.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent / "eda"))

from catboost_handmatchup import *  # noqa: F403

df = add_hand_matchup(load("train.csv"))
bc.FEATURES, bc.CAT_FEATURES = HAND_MATCHUP_FEATURES, HAND_MATCHUP_CAT_FEATURES
print(f"피처 {len(bc.FEATURES)}개, 그중 범주형 {len(bc.CAT_FEATURES)}개 (hand_matchup 포함)")

피처 45개, 그중 범주형 13개 (hand_matchup 포함)


## single-split 검증 (2019-23→24)

In [2]:
train_df, valid_df = bc.time_split(df, 2024)
model = bc.train_catboost(train_df, valid_df)
metrics = bc.evaluate(model, valid_df)
print(metrics)

{'n': 253507, 'r (실제 성공률)': np.float64(0.4861), 'brier': 0.247944, 'baseline_brier (r(1-r))': np.float64(0.249807), 'score (리더보드 산식)': np.float64(745.72), 'auc': 0.5488, 'calibration_slope': 1.1279}


## rolling OOT 검증 (fixed-iteration, 올바른 방법론)

2019-21→22, 2019-22→23, 2019-23→24 세 폴드를 모두 같은 iterations로 고정 학습해서 비교한다. 2023은 regime transition을 평가하는 stress fold라 baseline 자체도 붕괴(score=0)하므로 채택 판단 근거로 쓰지 않는다 — 2024(primary)와 2022 두 폴드가 판단 기준.

In [3]:
fold_results = bc.rolling_oot_evaluate_fixed(df, model.get_best_iteration() + 1)
for season, m in fold_results.items():
    print(season, m)

2022 {'n': 247472, 'r (실제 성공률)': np.float64(0.5289), 'brier': 0.243392, 'baseline_brier (r(1-r))': np.float64(0.249164), 'score (리더보드 산식)': np.float64(2316.31), 'auc': 0.5782, 'calibration_slope': 1.0722, 'weight': 0.2}
2023 {'n': 245525, 'r (실제 성공률)': np.float64(0.5), 'brier': 0.253115, 'baseline_brier (r(1-r))': np.float64(0.25), 'score (리더보드 산식)': 0.0, 'auc': 0.5279, 'calibration_slope': 0.1615, 'weight': 0.3}
2024 {'n': 253507, 'r (실제 성공률)': np.float64(0.4861), 'brier': 0.247926, 'baseline_brier (r(1-r))': np.float64(0.249807), 'score (리더보드 산식)': np.float64(752.89), 'auc': 0.5491, 'calibration_slope': 1.1142, 'weight': 0.5}
weighted {'brier': 0.248576, 'score': np.float64(839.71)}


baseline(hand_matchup 없이) 대비:

| 폴드 | baseline | +hand_matchup | 차이 |
| --- | ---: | ---: | ---: |
| 2022 (weight 0.2) | 2257.03 | 2316.31 | +59.28 |
| 2023 (weight 0.3, stress) | 0.0 | 0.0 | 판단 제외 |
| 2024 (weight 0.5, primary) | 734.49 | 752.89 | **+18.40** |
| weighted | 818.65 | 839.71 | +21.06 |

2022와 2024 둘 다 같은 방향으로 개선 → 노이즈가 아니라 실신호로 판단, 채택.

## 왜 hand_matchup이 먹히는가 — residual 분석

baseline(hand_matchup 없는 모델)의 예측 오차(`y - p`)를 `pitcher_hand x batter_hand` 조합별로 평균 내면, 뚜렷한 단조 패턴이 나온다. `residual_analysis.py`의 결과를 재현한다.

In [4]:
bc.FEATURES, bc.CAT_FEATURES = list(bc.FEATURES[:-1]), list(bc.CAT_FEATURES[:-1])  # hand_matchup 뺀 baseline
baseline_train, baseline_valid = bc.time_split(df, 2024)
baseline_model = bc.train_catboost(baseline_train, baseline_valid)
p = baseline_model.predict_proba(bc.to_pool(baseline_valid, with_label=False))[:, 1]
resid = baseline_valid[bc.TARGET].to_numpy() - p
import pandas as pd
v = baseline_valid.copy()
v["residual"] = resid
print(v.groupby("hand_matchup")["residual"].agg(["mean", "count"]).sort_values("mean"))

                  mean  count
hand_matchup                 
1_1          -0.022351  30493
2_2          -0.013677  92409
2_1          -0.002535  89268
1_2           0.004556  41337


`1_1`(둘 다 왼손 계열) 조합이 가장 크게 과대예측(-0.022)되고, `1_2` 조합은 부호가 반전(+0.005, 과소예측)된다 — baseline이 두 hand의 main effect만으로는 못 잡는, 조합 특이적 편향이 실제로 존재한다는 뜻. 이게 hand_matchup을 명시적 categorical로 추가했을 때 개선되는 메커니즘으로 보인다.

## 최종 모델 (전체 데이터 재학습, 제출용)

검증에서 찾은 iteration 수를 고정하고 2019~2024 전체로 재학습한다 (검증에 쓴 데이터로 다시 검증하는 실수 방지 — 이 프로젝트에서 CatBoost/NN 둘 다 한 번씩 저지른 적 있는 실수라 항상 확인).

In [5]:
bc.FEATURES, bc.CAT_FEATURES = HAND_MATCHUP_FEATURES, HAND_MATCHUP_CAT_FEATURES
full_model = bc.train_final_full(df, model.get_best_iteration() + 1)
bc.MODEL_DIR.mkdir(exist_ok=True)
full_model.save_model(str(bc.MODEL_DIR / "catboost_handmatchup.cbm"))
print(f"saved {bc.MODEL_DIR / 'catboost_handmatchup.cbm'}")

saved /Users/choehabin/Library/CloudStorage/OneDrive-개인/lg aimers/모델링/modeling/models/catboost_handmatchup.cbm


## 결론

`hand_matchup`(pitcher_hand x batter_hand)을 44개 기존 피처에 추가한 45피처 CatBoost 모델이 rolling OOT에서 baseline을 일관되게 앞섰다(primary 2024 기준 +18.40, weighted +21.06). 제출 패키지: [`submit_catboost_handmatchup/submit.zip`](../submit_catboost_handmatchup/submit.zip). `team_matchup`/`count_state`는 같은 방식으로 재확인해도 baseline 대비 확실히 나빠서 폐기 (HANDOFF.md 실험 로그).